# 04 — Evaluate Similarity: OD Model vs SD Model

Quantify how closely the CART-synthesised regression model reproduces the original regression model using:

1. **Euclidean distance** between estimated coefficient vectors  
   $$d(\hat{\beta}_{OD}, \hat{\beta}_{SD}) = \sqrt{\sum_{j=0}^{p} (\hat{\beta}_{OD,j} - \hat{\beta}_{SD,j})^2}$$

2. **Prediction drift** — MSE and MAE between predictions of the two models on a fresh, independent test set.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import json
import os
import glob
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
# ── Load Configuration ──────────────────────────────────────────────────────
config_path = os.path.join("..", "config", "config.json")
with open(config_path, "r") as f:
    config = json.load(f)

N_test   = config["simulation"]["N_test"]
p        = config["simulation"]["p"]
sigma_2  = config["simulation"]["sigma_2"]
test_seed = config["simulation"]["test_seed"]
rho_vals = config["parameters"]["rho"]
beta     = np.array(config["parameters"]["beta"])  # length p+1

print(f"Test-set config: N_test={N_test}, p={p}, sigma_2={sigma_2}, seed={test_seed}")
print(f"Rho values: {rho_vals}")
print(f"True beta:  {beta}")

In [ ]:
# ── Helper: generate an independent test dataset ───────────────────────────
def make_sigma(p, rho):
    """Equi-correlation covariance matrix (1 on diag, rho off-diag)."""
    Sigma = np.full((p, p), rho)
    np.fill_diagonal(Sigma, 1.0)
    return Sigma


def generate_test(N, p, rho, beta, sigma_2, seed=123):
    """Generate X ~ MVN(0, Sigma), y = Xβ + ε, return DataFrame."""
    rng = np.random.default_rng(seed)
    Sigma = make_sigma(p, rho)
    X = rng.multivariate_normal(mean=np.zeros(p), cov=Sigma, size=N)
    X_design = np.column_stack([np.ones(N), X])  # add intercept column
    epsilon = rng.normal(0, np.sqrt(sigma_2), size=N)
    y = X_design @ beta + epsilon

    df = pd.DataFrame(X, columns=[f"X{i+1}" for i in range(p)])
    df["y"] = y
    return df

In [ ]:
# ── Discover saved model files ─────────────────────────────────────────────
model_dir  = os.path.join("..", "models")
result_dir = os.path.join("..", "results")
os.makedirs(result_dir, exist_ok=True)

model_files = sorted(glob.glob(os.path.join(model_dir, "models_*.pkl")))
assert len(model_files) > 0, f"No model files found in {model_dir}"

print(f"Found {len(model_files)} model pair(s) to evaluate.")

## Evaluate Each Scenario

In [ ]:
# ── Evaluate each scenario ─────────────────────────────────────────────────
all_metrics = []

for mf in model_files:
    with open(mf, "rb") as f:
        bundle = pickle.load(f)

    lm_od    = bundle["lm_od"]
    lm_sd    = bundle["lm_sd"]
    scenario = bundle["scenario"]

    # Extract rho from scenario string, e.g. 'N1000_p10_rho0.3'
    rho = float(scenario.split("rho")[-1])

    print(f"\n{'='*60}")
    print(f"Scenario: {scenario}  (rho = {rho})")
    print(f"{'='*60}")

    # ── (a) Euclidean distance between coefficient vectors ─────────────────
    beta_od = np.concatenate([[lm_od.intercept_], lm_od.coef_])
    beta_sd = np.concatenate([[lm_sd.intercept_], lm_sd.coef_])
    eucl_dist = np.linalg.norm(beta_od - beta_sd)
    print(f"  Euclidean distance (coefficients): {eucl_dist:.6f}")

    # ── (b) Prediction-based metrics on fresh test data ────────────────────
    test_data = generate_test(N_test, p, rho, beta, sigma_2, seed=test_seed)
    X_test = test_data.drop(columns="y")

    pred_od = lm_od.predict(X_test)
    pred_sd = lm_sd.predict(X_test)

    mse_pred = mean_squared_error(pred_od, pred_sd)
    mae_pred = mean_absolute_error(pred_od, pred_sd)
    print(f"  Prediction MSE (OD vs SD): {mse_pred:.6f}")
    print(f"  Prediction MAE (OD vs SD): {mae_pred:.6f}")

    all_metrics.append({
        "scenario": scenario,
        "rho": rho,
        "euclidean_dist": eucl_dist,
        "prediction_mse": mse_pred,
        "prediction_mae": mae_pred,
    })

## Summary Table

In [ ]:
metrics_df = pd.DataFrame(all_metrics)
display(metrics_df)

# Save to CSV
out_path = os.path.join(result_dir, "evaluation_metrics.csv")
metrics_df.to_csv(out_path, index=False)
print(f"\nMetrics saved to: {out_path}")

## Visualisations

In [ ]:
# ── Plot 1: Euclidean distance vs rho ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Euclidean distance
axes[0].bar(metrics_df["rho"].astype(str), metrics_df["euclidean_dist"],
            color="steelblue", edgecolor="black")
axes[0].set_xlabel("Correlation (ρ)")
axes[0].set_ylabel("Euclidean Distance")
axes[0].set_title("Coefficient Distance: OD vs SD")

# (b) Prediction MSE
axes[1].bar(metrics_df["rho"].astype(str), metrics_df["prediction_mse"],
            color="coral", edgecolor="black")
axes[1].set_xlabel("Correlation (ρ)")
axes[1].set_ylabel("MSE")
axes[1].set_title("Prediction MSE: OD vs SD")

# (c) Prediction MAE
axes[2].bar(metrics_df["rho"].astype(str), metrics_df["prediction_mae"],
            color="seagreen", edgecolor="black")
axes[2].set_xlabel("Correlation (ρ)")
axes[2].set_ylabel("MAE")
axes[2].set_title("Prediction MAE: OD vs SD")

plt.tight_layout()
plt.savefig(os.path.join(result_dir, "plot_evaluation_metrics.png"), dpi=150)
plt.show()
print("Plot saved to ../results/plot_evaluation_metrics.png")

In [ ]:
# ── Plot 2: Coefficient comparison (OD vs SD) per scenario ─────────────────
for mf in model_files:
    with open(mf, "rb") as f:
        bundle = pickle.load(f)

    lm_od    = bundle["lm_od"]
    lm_sd    = bundle["lm_sd"]
    scenario = bundle["scenario"]

    terms = ["β₀"] + [f"β{i+1}" for i in range(p)]
    beta_od = np.concatenate([[lm_od.intercept_], lm_od.coef_])
    beta_sd = np.concatenate([[lm_sd.intercept_], lm_sd.coef_])

    x_pos = np.arange(len(terms))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x_pos - width/2, beta_od, width, label="OD", color="steelblue",
           edgecolor="black")
    ax.bar(x_pos + width/2, beta_sd, width, label="SD", color="coral",
           edgecolor="black")
    ax.axhline(y=1.0, color="grey", linestyle="--", linewidth=0.8,
               label="True β = 1")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(terms)
    ax.set_ylabel("Estimated Coefficient")
    ax.set_title(f"Coefficient Comparison — {scenario}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(result_dir, f"plot_coef_{scenario}.png"), dpi=150)
    plt.show()

print("\n[DONE] Evaluation complete.")